# Agentic Pattern - Parallelization
Parallelization involves executing multiple components, such as LLM calls, tool usages, or even entire sub-agents, concurrently 

[![Agentic Pattern - Parallelization presentation](https://img.youtube.com/vi/?/0.jpg)](https://youtu.be/?) 

https://yt3.googleusercontent.com/zADzdtnNlOO4rCQp3Dqfmdpo_jTFFSL-Ti3M042TDE2dwEO-WqY2Uk-LSIYwI4vaY0AATDz1Vw=s160-c-k-c0x00ffffff-no-rj

<br />
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<a href="https://www.youtube.com/@dayonedev" target="_new">
  <img align="center" src="https://img.shields.io/youtube/channel/views/UCiLziPE9aPxCouSsX0lJ--A" />
</a>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<a href="https://linkedin.com/in/krishnamanchikalapudi" target="_new">
  <img align="center" src="https://img.shields.io/badge/linkedin-%230077B5.svg?style=for-the-badge&logo=linkedin&logoColor=white" />
</a>
<br/>

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 7.27 ms, sys: 9.79 ms, total: 17.1 ms
Wall time: 1.03 s


In [2]:
%pip install -U -q langchain langchain-ollama ipython-autotime --use-deprecated=legacy-resolver

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
time: 147 μs (started: 2025-09-18 22:12:08 -07:00)


## Variables

In [3]:
my_model_ollama = "llama3.2"

time: 205 μs (started: 2025-09-18 22:12:08 -07:00)


## Initialize OLLAM service
OLLAMA inference client is initialized to interact with the OLLAMA API for generating responses from the specified model.

In [4]:
from langchain_ollama.llms import OllamaLLM

llm_client = OllamaLLM(
    model=my_model_ollama,
    base_url="http://localhost:11434",
    headers={"Content-Type": "application/json"},
    stream=True,
    temperature=0.7,
)

llm_client

OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')

time: 1.53 s (started: 2025-09-18 22:12:08 -07:00)


### TEST llm_client with a simple prompt

In [5]:
response = llm_client.invoke("What is Agentic Pattern - Parallelization")

print(f"{response}")

Agentic Pattern - Parallelization is a design pattern that refers to the use of parallel processing techniques to execute multiple tasks concurrently, leveraging the power of multi-core processors and distributed computing systems.

The term "Agentic" comes from the word "agent", which implies an autonomous entity that can act independently. In this context, the agent is the program or task that is being executed in parallel.

Parallelization is a technique used to speed up computation by dividing tasks into smaller sub-tasks and executing them concurrently on multiple processing units. This approach can significantly improve performance, especially for computationally intensive tasks like scientific simulations, data analysis, and machine learning.

The Agentic Pattern - Parallelization involves the following key aspects:

1. **Task decomposition**: Breaking down complex tasks into smaller, independent sub-tasks that can be executed concurrently.
2. **Parallel execution**: Executing m

## Define Independent Chains
These three chains represent distinct tasks that can be executed in parallel.

In [6]:
from langchain_core.runnables import Runnable
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages(
        [("system", "Summarize the following topic concisely:"), ("user", "{topic}")]
    )
    | llm_client
    | StrOutputParser()
)

summarize_chain

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Summarize the following topic concisely:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic}'), additional_kwargs={})])
| OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
| StrOutputParser()

time: 20.9 ms (started: 2025-09-18 22:12:18 -07:00)


In [7]:
questions_chain: Runnable = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Generate three interesting questions about the following topic:",
            ),
            ("user", "{topic}"),
        ]
    )
    | llm_client
    | StrOutputParser()
)

questions_chain

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Generate three interesting questions about the following topic:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic}'), additional_kwargs={})])
| OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
| StrOutputParser()

time: 6.56 ms (started: 2025-09-18 22:12:18 -07:00)


In [8]:
terms_chain: Runnable = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Identify 5-10 key terms from the following topic, separated by commas:",
            ),
            ("user", "{topic}"),
        ]
    )
    | llm_client
    | StrOutputParser()
)

terms_chain

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Identify 5-10 key terms from the following topic, separated by commas:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic}'), additional_kwargs={})])
| OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
| StrOutputParser()

time: 5.71 ms (started: 2025-09-18 22:12:18 -07:00)


## Parallel Execution
RunnableParallel will run all tasks concurrently

1. Define the block of tasks to run in parallel. The results of these, along with the original topic, will be fed into the next step.

In [9]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

parallel_tasks = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough(),  # Pass the original topic
    }
)

parallel_tasks

{
  summary: ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Summarize the following topic concisely:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic}'), additional_kwargs={})])
           | OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
           | StrOutputParser(),
  questions: ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Generate three interesting questions about the following topic:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables

time: 9.33 ms (started: 2025-09-18 22:12:18 -07:00)


2. Define the final synthesis prompt which will combine the parallel results.

In [10]:
synthesis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Based on the following information:
     Summary: {summary}
     Related Questions: {questions}
     Key Terms: {key_terms}
     Write a cohesive final answer that combines all three views in a clear way.""",
        ),
        ("user", "Original topic: {topic}"),
    ]
)

synthesis_prompt

ChatPromptTemplate(input_variables=['key_terms', 'questions', 'summary', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['key_terms', 'questions', 'summary'], input_types={}, partial_variables={}, template='Based on the following information:\n     Summary: {summary}\n     Related Questions: {questions}\n     Key Terms: {key_terms}\n     Write a cohesive final answer that combines all three views in a clear way.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Original topic: {topic}'), additional_kwargs={})])

time: 2.79 ms (started: 2025-09-18 22:12:18 -07:00)


In [11]:
synthesis_chain = synthesis_prompt | llm_client | StrOutputParser()

synthesis_chain

ChatPromptTemplate(input_variables=['key_terms', 'questions', 'summary', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['key_terms', 'questions', 'summary'], input_types={}, partial_variables={}, template='Based on the following information:\n     Summary: {summary}\n     Related Questions: {questions}\n     Key Terms: {key_terms}\n     Write a cohesive final answer that combines all three views in a clear way.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Original topic: {topic}'), additional_kwargs={})])
| OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
| StrOutputParser()

time: 1.01 ms (started: 2025-09-18 22:12:18 -07:00)


 3. Construct the full chain by piping the parallel results directly into the synthesis prompt, followed by the LLM and output parser.


In [12]:
full_chain = parallel_tasks | synthesis_chain

full_chain

{
  summary: ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Summarize the following topic concisely:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic}'), additional_kwargs={})])
           | OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
           | StrOutputParser(),
  questions: ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Generate three interesting questions about the following topic:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables

time: 3.39 ms (started: 2025-09-18 22:12:18 -07:00)


## Run the Chain
 Asynchronously invokes the parallel processing chain with a specific topic and prints the synthesized result.

In [13]:
my_topic = "The history of space exploration"

result = full_chain.invoke(my_topic)

print(f"\n--- Running Parallelization Pattern and Result --- \n")
print(f"{result}")


--- Running Parallelization Pattern and Result --- 

The History of Space Exploration: A Comprehensive Overview

The history of space exploration is a rich and fascinating field that has captivated human imagination for decades. From the early years of satellite launches to the present day, space agencies around the world have pushed the boundaries of what is possible, driven by an insatiable curiosity about the universe.

**Early Years (1950s-1960s)**

The journey into space began in 1957 with the launch of Sputnik 1, the first artificial satellite, by the Soviet Union. This achievement marked a significant milestone in the history of space exploration and sparked a response from the United States, which launched Project Mercury to catch up on the technological advancements made by its rival. The first American astronauts, Alan Shepard and John Glenn, became the first Americans in space, followed by Yuri Gagarin, who orbited Earth in 1961.

**Moon Landing (1969)**

The highlight of t

<br />
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
Subscribe to <a href="https://www.youtube.com/@dayonedev" target="_new">
  <img align="center" src="https://img.shields.io/youtube/channel/views/UCiLziPE9aPxCouSsX0lJ--A" />
</a>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
Connect me at <a href="https://linkedin.com/in/krishnamanchikalapudi" target="_new">
  <img align="center" src="https://img.shields.io/badge/linkedin-%230077B5.svg?style=for-the-badge&logo=linkedin&logoColor=white" />
</a>
<br/>